# Avatar thật cho staging với 0đ (Colab T4 + HLS)

Cell dưới dựng **toàn bộ** chuỗi avatar trên một máy Colab, không Docker:

`LiveTalking (rtcpush) → MediaMTX (HLS) → cloudflared (HTTPS) → AI service (FastAPI) → cloudflared (HTTPS)`

Lý do không dùng WebRTC thuần của LiveTalking: kênh media là UDP đi thẳng từ máy chủ tới trình duyệt,
Colab và tunnel HTTP (ngrok/cloudflared) không chuyển được UDP. Xem `docs/AVATAR-HLS-FREE.md`.

**Điều kiện**
- Runtime → Change runtime type → **Hardware accelerator: T4 GPU** (free).
- Lần chạy đầu ~20–30 phút: tải 300MB weights + build avatar pack. Lần sau nhanh hơn nếu chưa mất disk.
- Không đóng tab; cell keepalive bên dưới giữ session (Colab free cắt sau ~12h).

In [ ]:
# Repo public nên Colab clone được không cần token. Đổi BE_BRANCH sang nhánh chưa lên dev.
BE_BRANCH = "dev"
!nvidia-smi -L
!git clone -q -b {BE_BRANCH} https://github.com/PNV-do-an/AI-Interview-Platform-BE.git /content/be
# KHÔNG chạy "git submodule update --init" ở đây: vendored/LiveTalking là gitlink mà repo BE
# chưa có .gitmodules, lệnh đó báo lỗi và vẫn không có nguồn. colab_avatar.sh tự clone
# lipku/LiveTalking (pin commit đã kiểm chứng) ở bước 0/8.
!ls -l /content/be/scripts/colab_avatar.sh

In [ ]:
# Cài deps -> tải weights -> build avatar pack -> MediaMTX -> LiveTalking rtcpush -> tunnel HLS -> AI service -> tunnel AI.
# FE_ORIGIN là origin staging; AI service chỉ mở CORS cho origin này.
!BE_ROOT=/content/be FE_ORIGIN=https://aiinterview-vn.vercel.app bash /content/be/scripts/colab_avatar.sh

In [ ]:
# In hai URL và xác nhận AI service tự khai báo luồng avatar qua /health.
!cat /content/avatar_urls.txt
!curl -s $(cat /content/ai_url.txt)/health | python3 -m json.tool

In [ ]:
# Giữ session Colab: chạy cell này ở cuối và để nguyên (thread nền ping mỗi 9 phút).
import threading, time

def _keep():
    from google.colab import output
    while True:
        try:
            output.eval_js('new Date().toString()')
        except Exception:
            pass
        time.sleep(9 * 60)

threading.Thread(target=_keep, daemon=True).start()
print('keepalive đang chạy nền (ping mỗi 9 phút)')

## Nối vào staging

1. Vercel → project `ai-interview` → Settings → Environment Variables → `REACT_APP_AI_SERVICE_URL` =
   giá trị trong `/content/ai_url.txt` (type Production) → Redeploy. Biến này là build-time của CRA
   nên **mỗi lần URL đổi phải redeploy**; `mediaUrl` thì FE đọc động từ `/health`, không cần redeploy.
2. Mở https://aiinterview-vn.vercel.app/mock-interviews → chọn **LiveTalking Avatar (Real)** → vào phòng.
   Trong phòng: footer hiện `• Giọng đọc: avatar LiveTalking` và khung video có hình là đã ăn thật.
3. Bấm **Bật tiếng avatar** một lần (Chrome chặn tự phát có tiếng cho tới khi người dùng tương tác).

## Khi phòng tự chuyển về giọng đọc trình duyệt

| Chữ hiện trong phòng | Nghĩa | Kiểm tra |
| --- | --- | --- |
| `Máy chủ avatar không phản hồi` | AI service gọi LiveTalking không được | `tail /content/logs/livetalking.log` |
| `Luồng video avatar bị đứt (...)` | HLS lỗi giữa phiên (Colab mất/tunnel đổi) | `tail /content/logs/mediamtx.log`, chạy lại cell 3 |
| `AI service không phản hồi` | tunnel AI chết | `curl $(cat /content/ai_url.txt)/health` |

Log: `/content/logs/{livetalking,ai,mediamtx,cf-ai,cf-hls}.log`.